# 06 — End-to-end digital-twin run

Runs the full LangGraph loop against the 14-day test slice and plots cumulative cost, savings trajectory, and detected/planned events. Expect ~2 min for a 24-h simulation on CPU.

In [ ]:
import subprocess, sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
# Run the twin for 48 simulated hours and inspect the log.
subprocess.run([sys.executable, "-m", "aerogrid.sim.digital_twin",
                "--use-test-window", "--hours", "48"], check=True)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, json
from aerogrid.config import RUN_LOG_PATH
rows = [json.loads(l) for l in RUN_LOG_PATH.read_text().splitlines() if l.strip()]
log = pd.DataFrame(rows)
log["now"] = pd.to_datetime(log["now"])
fig, (ax1, ax2) = plt.subplots(2,1, figsize=(12,6), sharex=True)
ax1.plot(log["now"], log["cumulative_cost"], label="optimizer")
ax1.plot(log["now"], log["cumulative_baseline_cost"], label="naive baseline", linestyle="--")
ax1.set_ylabel("$"); ax1.legend(); ax1.set_title("cumulative cost")
ax2.plot(log["now"], log["realized_price"], "k-", lw=0.7)
ax2.set_ylabel("realized LBMP ($/MWh)"); ax2.set_xlabel("time")
for reason, sub in log[log["replan_reason"].notna()].iterrows():
    ax2.axvline(sub["now"], color="red", alpha=0.3, linewidth=0.5)
plt.tight_layout(); plt.show()
print(f"n ticks: {len(log)}  replans: {log['replan_reason'].notna().sum()}  final savings: "
      f"{(1 - log['cumulative_cost'].iloc[-1] / max(log['cumulative_baseline_cost'].iloc[-1], 1e-6))*100:.1f}%")